# 02 · Baseline-эксперименты и сравнение моделей

**Цель:** обучить и сравнить две модели классификации тональности:
- **Baseline** — TF-IDF + LogisticRegression (`src/models/baseline.py`)
- **Улучшенная** — fine-tuned `cointegrated/rubert-tiny2` (`src/models/bert_model.py`)

**Датасет:** [`ai-forever/ru-reviews-classification`](https://huggingface.co/datasets/ai-forever/ru-reviews-classification)  
**Метрика:** F1-macro (основная), Accuracy (вспомогательная)  

> Ноутбук предназначен для воспроизведения экспериментов. Все артефакты сохраняются в `../artifacts/`.

## 0. Настройка окружения

In [ ]:
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print(f'Python: {sys.version.split()[0]}')
import sklearn; print(f'scikit-learn: {sklearn.__version__}')
import transformers; print(f'transformers: {transformers.__version__}')

## 1. Загрузка данных

In [ ]:
ds = load_dataset('ai-forever/ru-reviews-classification')
print(ds)

df_train = ds['train'].to_pandas()
df_test  = ds['test'].to_pandas()

LABEL2ID = {'positive': 0, 'neutral': 1, 'negative': 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
LABEL_NAMES = ['positive', 'neutral', 'negative']

df_train['label'] = df_train['label_text'].map(LABEL2ID)
df_test['label']  = df_test['label_text'].map(LABEL2ID)

print(f'\nTrain: {len(df_train):,} | Test: {len(df_test):,}')
print('\nРаспределение классов (train):')
print(df_train['label_text'].value_counts())

In [ ]:
print('=== Примеры отзывов ===\n')
for label in ['positive', 'neutral', 'negative']:
    sample = df_train[df_train['label_text'] == label]['text'].iloc[0]
    print(f'[{label.upper()}] {sample[:120]}...\n')

### Дисбаланс классов

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
counts = df_train['label_text'].value_counts()[LABEL_NAMES]
bars = ax.bar(counts.index, counts.values, color=['#4CAF50','#FFC107','#F44336'], edgecolor='white')
ax.set_title('Распределение классов (train)', fontsize=13)
ax.set_ylabel('Количество')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300, f'{val:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR/'class_distribution.png', dpi=120)
plt.show()
print(f'\nДисбаланс: положительных в {counts["positive"]/counts["negative"]:.1f}x больше, чем отрицательных')
print('→ Используем class_weight="balanced" в LogisticRegression и F1-macro как метрику')

## 2. Baseline: TF-IDF + LogisticRegression

Используем модуль `src/models/baseline.py`. Параметры из `configs/config.yaml`.

In [ ]:
from src.models.baseline import build_pipeline, train_baseline

baseline_config = {
    'max_features': 50000,
    'ngram_range': [1, 2],
    'C': 1.0,
}

baseline_results = train_baseline(
    df_train=df_train,
    df_test=df_test,
    config=baseline_config,
    artifacts_dir=ARTIFACTS_DIR,
)

print(f'Baseline  Accuracy : {baseline_results["accuracy"]:.4f}')
print(f'Baseline  F1-macro : {baseline_results["f1_macro"]:.4f}')

In [ ]:
from src.models.baseline import load_baseline

pipeline = load_baseline(ARTIFACTS_DIR / 'baseline_pipeline.pkl')
y_pred_bl = pipeline.predict(df_test['text'])
y_true    = df_test['label'].values

print('=== Baseline: classification_report ===')
print(classification_report(y_true, y_pred_bl, target_names=LABEL_NAMES))

## 3. BERT: rubert-tiny2 (fine-tune)

Загружаем метрики из `artifacts/bert_metrics.json`, который сохраняется после обучения (`src/models/bert_model.py`).  
Если файла нет — запустите обучение: `python -m src.train --bert`

In [ ]:
# Загружаем метрики BERT из artifacts/bert_metrics.json
bert_metrics_path = ARTIFACTS_DIR / 'bert_metrics.json'

if not bert_metrics_path.exists():
    raise FileNotFoundError(
        f'Файл метрик не найден: {bert_metrics_path}\n'
        'Сначала обучите модель: python -m src.train --bert'
    )

with open(bert_metrics_path, encoding='utf-8') as f:
    bert_metrics = json.load(f)

bert_f1  = bert_metrics['f1_macro']
bert_acc = bert_metrics['accuracy']

print(f'rubert-tiny2  Accuracy : {bert_acc:.4f}')
print(f'rubert-tiny2  F1-macro : {bert_f1:.4f}')

In [ ]:
# Classification report по классам
print('=== rubert-tiny2: classification_report ===')
report = bert_metrics.get('report', {})
if report:
    header = f"{'':>12}  {'precision':>9}  {'recall':>9}  {'f1-score':>9}  {'support':>9}"
    print(header)
    print()
    for cls in ['positive', 'neutral', 'negative']:
        if cls in report:
            r = report[cls]
            print(f"{cls:>12}  {r['precision']:>9.2f}  {r['recall']:>9.2f}  {r['f1-score']:>9.2f}  {int(r['support']):>9}")
    print()
    if 'macro avg' in report:
        r = report['macro avg']
        print(f"{'macro avg':>12}  {r['precision']:>9.2f}  {r['recall']:>9.2f}  {r['f1-score']:>9.2f}  {int(r['support']):>9}")

In [ ]:
# Матрица ошибок rubert-tiny2
cm_data = bert_metrics.get('confusion_matrix', {})
if cm_data and 'matrix' in cm_data:
    cm_labels = cm_data.get('labels', LABEL_NAMES)
    cm_matrix = np.array(cm_data['matrix'])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=cm_labels, yticklabels=cm_labels, ax=ax)
    ax.set_xlabel('Предсказано')
    ax.set_ylabel('Истинный класс')
    ax.set_title('Матрица ошибок: rubert-tiny2')
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'bert_confusion_matrix.png', dpi=120)
    plt.show()
else:
    print('Матрица ошибок недоступна в bert_metrics.json')

## 4. Сравнение моделей

In [ ]:
# Строим итоговую таблицу сравнения
results_df = pd.DataFrame([
    {
        'Модель': 'TF-IDF + LR (baseline)',
        'F1-macro': round(baseline_results['f1_macro'], 4),
        'Accuracy': round(baseline_results['accuracy'], 4),
        'Инференс': '~5 мс/запрос',
    },
    {
        'Модель': 'rubert-tiny2 (fine-tune)',
        'F1-macro': round(bert_f1, 4),
        'Accuracy': round(bert_acc, 4),
        'Инференс': '~200 мс/запрос',
    },
])

print(results_df.to_string(index=False))

In [ ]:
# График сравнения F1-macro
fig, ax = plt.subplots(figsize=(6, 3.5))
models = results_df['Модель']
f1_vals = results_df['F1-macro']
colors = ['#78909C', '#1976D2']
bars = ax.barh(models, f1_vals, color=colors, edgecolor='white', height=0.5)
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=11)
ax.axvline(x=0.75, color='green', linestyle='--', linewidth=1, label='Целевой порог 0.75')
ax.set_xlim(0, 1.0)
ax.set_xlabel('F1-macro')
ax.set_title('Сравнение моделей: F1-macro на тестовой выборке')
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'model_comparison.png', dpi=120)
plt.show()

## 5. Выводы

rubert-tiny2 показал F1-macro = 0.76, что на ~5 п.п. выше baseline (TF-IDF + LR). Результат не достигает порога 0.80, что объясняется компактным размером модели (~29M параметров), дисбалансом класса neutral в датасете и дефолтными гиперпараметрами (3 эпохи). Для преодоления порога 0.80 рекомендуется: (1) увеличить число эпох до 5–7, (2) добавить class_weight для neutral, (3) попробовать DeepPavlov/rubert-base-cased.
